## Feature Engineering  

## 1. Objetivo

Criar variáveis que ajudam o modelo a entender o tempo

## 2 Importação das Bibliotecas

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import sqlite3

from IPython.display import Markdown
from IPython.core.display import HTML
import math

import glob
import warnings
warnings.filterwarnings('ignore')

## 3. Configuraçãões Iniciais

In [2]:
# Cor principal do projeto
PRIMARY_COLOR = "#50e550"
SECONDARY_COLORS = sns.light_palette(PRIMARY_COLOR, n_colors=5)

# Estilo geral
sns.set_theme(style="whitegrid")

# Tamanho padrão
plt.rcParams['figure.figsize'] = (10, 6)

# Fonte
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

## 4. Carregamento e Leitura dos Dados


In [3]:
df_demand = pd.read_parquet('../data/processed/df_demand.parquet')

df_demand.head()

,product_id,date,sales
16,001b72dfd63e9833e8c02742adf472e3,2017-02-15,2
17,001b72dfd63e9833e8c02742adf472e3,2017-02-22,1
18,001b72dfd63e9833e8c02742adf472e3,2017-03-01,1
19,001b72dfd63e9833e8c02742adf472e3,2017-07-02,1
20,001b72dfd63e9833e8c02742adf472e3,2017-07-08,2


## 5. Preparação inicial

In [4]:
df = df_demand.copy()

df['date'] = pd.to_datetime(df['date'])

df = df.sort_values(['product_id', 'date'])

## 6. Features de tempo

Capturam padrões como sazonalidade semanal e mensal.

In [5]:
df['day_of_week'] = df['date'].dt.dayofweek
df['day'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df['week_of_year'] = df['date'].dt.isocalendar().week

## 7. Features de Lag

Capturam dependência temporal (o passado influencia o futuro).

In [6]:
df['lag_1'] = df.groupby('product_id')['sales'].shift(1)
df['lag_7'] = df.groupby('product_id')['sales'].shift(7)
df['lag_14'] = df.groupby('product_id')['sales'].shift(14)

## 8. Médias Móveis

Suavizam a série e ajudam a capturar tendência.

In [7]:
df['rolling_mean_7'] = (
    df.groupby('product_id')['sales']
      .shift(1)
      .rolling(7)
      .mean()
)

df['rolling_mean_14'] = (
    df.groupby('product_id')['sales']
      .shift(1)
      .rolling(14)
      .mean()
)

In [8]:
df['rolling_std_7'] = (
    df.groupby('product_id')['sales']
      .shift(1)
      .rolling(7)
      .std()
)

## 9. Tratamento de valores faltantes (gerados pelas features)

Tratamento de NaNs

As features de lag e rolling geram valores faltantes no início da série.
Esses registros são removidos para garantir consistência na modelagem.

In [9]:
df_features = df.dropna()

In [10]:

df_features.info()

<class 'pandas.DataFrame'>
Index: 14470 entries, 133 to 89051
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype        
---  ------           --------------  -----        
 0   product_id       14470 non-null  str          
 1   date             14470 non-null  datetime64[s]
 2   sales            14470 non-null  int64        
 3   day_of_week      14470 non-null  int32        
 4   day              14470 non-null  int32        
 5   month            14470 non-null  int32        
 6   year             14470 non-null  int32        
 7   week_of_year     14470 non-null  UInt32       
 8   lag_1            14470 non-null  float64      
 9   lag_7            14470 non-null  float64      
 10  lag_14           14470 non-null  float64      
 11  rolling_mean_7   14470 non-null  float64      
 12  rolling_mean_14  14470 non-null  float64      
 13  rolling_std_7    14470 non-null  float64      
dtypes: UInt32(1), datetime64[s](1), float64(6), int32(4), int64(1), str(

## 10. Salvar dataset

In [11]:
df_features.to_parquet('../data/processed/df_features.parquet')

## 11. Conclusão

Foram criadas features relevantes que capturam padrões temporais e comportamento histórico da demanda.

Essas variáveis são fundamentais para modelos de previsão, pois permitem:
- capturar sazonalidade
- entender dependência temporal
- reduzir ruído da série

O dataset final será utilizado na etapa de modelagem.